# Deepfake Group 31 - Evaluation
This notebook loads the best checkpoint, evaluates test performance, and reports confidence-based risk for predictions.

In [ ]:
from google.colab import drive
import os, random, numpy as np, torch

drive.mount('/content/drive')
BASE_DIR = '/content/drive/MyDrive/Deepfake_Group31'
SEED = 42
os.makedirs(BASE_DIR, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print(f"BASE_DIR set to: {BASE_DIR}")

## Install required dependencies
Install runtime dependencies for evaluation and visualization.

In [ ]:
!pip install -q torch torchvision facenet-pytorch opencv-python scikit-learn matplotlib tqdm wandb seaborn

## Import libraries for evaluation
Load model, data, plotting, and metric utilities.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split, Subset
from torchvision import transforms, models
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, precision_score, recall_score, roc_curve, confusion_matrix

## Define dataset and test loader
Recreate the dataset pipeline and deterministic split to evaluate the test partition.

In [ ]:
class DeepfakeImageDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.transform = transform
        self.samples = []
        class_map = {'real': 0, 'fake': 1}
        for class_name, label in class_map.items():
            class_dir = os.path.join(root_dir, class_name)
            if not os.path.isdir(class_dir):
                continue
            for file_name in sorted(os.listdir(class_dir)):
                if file_name.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.webp')):
                    self.samples.append((os.path.join(class_dir, file_name), label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        image_path, label = self.samples[idx]
        image = Image.open(image_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor([label], dtype=torch.float32), image_path

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

full_dataset = DeepfakeImageDataset(os.path.join(BASE_DIR, 'data'), transform=transform)
if len(full_dataset) == 0:
    raise ValueError('Dataset not found. Ensure BASE_DIR/data/{real,fake} is populated.')

target_total = 10000
if len(full_dataset) >= target_total:
    g = torch.Generator().manual_seed(SEED)
    sampled_idx = torch.randperm(len(full_dataset), generator=g)[:target_total].tolist()
    working_dataset = Subset(full_dataset, sampled_idx)
    split_lengths = [8000, 1000, 1000]
else:
    n = len(full_dataset)
    train_len = int(0.8 * n)
    val_len = int(0.1 * n)
    split_lengths = [train_len, val_len, n - train_len - val_len]
    working_dataset = full_dataset

g = torch.Generator().manual_seed(SEED)
_, _, test_dataset = random_split(working_dataset, split_lengths, generator=g)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

## Recreate model architecture and load checkpoint
Build the same detector structure used in training and load `best_model.pth`.

In [ ]:
class DeepfakeDetector(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        for idx in range(5):
            for param in backbone.features[idx].parameters():
                param.requires_grad = False
        backbone.classifier = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(1280, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, 1),
            nn.Sigmoid(),
        )
        self.model = backbone

    def forward(self, x):
        return self.model(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = DeepfakeDetector().to(device)
checkpoint_path = os.path.join(BASE_DIR, 'checkpoints', 'best_model.pth')
checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f'Loaded checkpoint from: {checkpoint_path}')

## Evaluate metrics on the test set
Compute Accuracy, F1-Score, AUC-ROC, Precision, and Recall.

In [ ]:
all_targets, all_probs = [], []
with torch.no_grad():
    for images, labels, _ in test_loader:
        images = images.to(device)
        probs = model(images).squeeze(1).cpu().numpy()
        all_probs.extend(probs.tolist())
        all_targets.extend(labels.squeeze(1).cpu().numpy().astype(int).tolist())

all_targets = np.array(all_targets)
all_probs = np.array(all_probs)
all_preds = (all_probs >= 0.5).astype(int)

metrics = {
    'accuracy': accuracy_score(all_targets, all_preds),
    'f1_score': f1_score(all_targets, all_preds),
    'auc_roc': roc_auc_score(all_targets, all_probs),
    'precision': precision_score(all_targets, all_preds),
    'recall': recall_score(all_targets, all_preds),
}

for key, value in metrics.items():
    print(f'{key}: {value:.4f}')

## Plot ROC curve and confusion matrix
Save both evaluation plots to Google Drive under the project `outputs` directory.

In [ ]:
outputs_dir = os.path.join(BASE_DIR, 'outputs')
os.makedirs(outputs_dir, exist_ok=True)
roc_path = os.path.join(outputs_dir, 'roc_curve.png')
cm_path = os.path.join(outputs_dir, 'confusion_matrix.png')

fpr, tpr, _ = roc_curve(all_targets, all_probs)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"AUC = {metrics['auc_roc']:.4f}")
plt.plot([0, 1], [0, 1], linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig(roc_path, dpi=200)
plt.show()

cm = confusion_matrix(all_targets, all_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['REAL', 'FAKE'], yticklabels=['REAL', 'FAKE'])
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.savefig(cm_path, dpi=200)
plt.show()

print(f'ROC curve saved to: {roc_path}')
print(f'Confusion matrix saved to: {cm_path}')

## Define confidence-based prediction helper
`predict_with_confidence()` returns label, confidence, and risk level using thresholds LOW `<0.3`, MEDIUM `0.3-0.6`, HIGH `>0.6`.

In [ ]:
def predict_with_confidence(model, image_tensor, device):
    model.eval()
    with torch.no_grad():
        score = float(model(image_tensor.unsqueeze(0).to(device)).item())

    label = 'FAKE' if score >= 0.5 else 'REAL'
    confidence = score

    if confidence < 0.3:
        risk_level = 'LOW'
    elif confidence <= 0.6:
        risk_level = 'MEDIUM'
    else:
        risk_level = 'HIGH'

    return {'label': label, 'confidence': confidence, 'risk_level': risk_level}

## Run inference on 5 test images
Print formatted confidence and risk outputs for five samples from the test split.

In [ ]:
print('Sample predictions on 5 test images:')
for i in range(min(5, len(test_dataset))):
    image_tensor, target, image_path = test_dataset[i]
    result = predict_with_confidence(model, image_tensor, device)
    gt = 'FAKE' if int(target.item()) == 1 else 'REAL'
    print(
        f"[{i+1}] file={os.path.basename(image_path)} | ground_truth={gt} | "
        f"pred={result['label']} | confidence={result['confidence']:.4f} | risk={result['risk_level']}"
    )